# Этап 3. Rule-based baseline — «красные флаги»

**Цель:** собрать набор правил, которыми реально пользуются комплаенс-отделы,
и получить честную точку отсчёта, которую дальше будет обходить ML-модель.

## Зачем правила, если мы всё равно будем строить XGBoost

| Причина | Что это значит на практике |
|---|---|
| **Точка отсчёта** | Модель обязана быть лучше правил. Если recall 40 % при 5 000 алертов в день хуже, чем 35 % при 300 алертах у правил — модель не окупается |
| **Объяснимость** | Регулятору и аналитику понятно «сработало правило R2: два платежа по 9 500 за сутки», а не «SHAP = 0.37» |
| **Закон** | Часть правил обязательна по закону (порог $10 000, страны из «чёрного списка») — их нельзя заменить моделью |
| **Гибрид** | В живом банке: правила отсекают очевидное, модель приоритизирует очередь алертов |

## Как измеряем

* **Recall (полнота)** — какую долю реального отмывания мы поймали;
* **Precision (точность)** — какая доля алертов действительно про отмывание;
* **Lift** — во сколько раз точнее, чем «проверять всё подряд»;
* **Алертов в день** — сколько работы это создаёт команде;
* **Recall по типологиям** — какие схемы мы ловим, а какие слепы.

`accuracy` не считаем вообще: при 0.1 % отмываний модель, которая вообще ничего
не нашла, даёт accuracy 99.9 %.

План:
1. Загрузка матрицы признаков (val + test)
2. Справочник правил и как они считаются
3. Метрики каждого правила
4. Стратегия «хотя бы одно правило»
5. Оценка риска: сколько правил сработало → выбор порога
6. Recall по типологиям: где наши слепые зоны
7. Нагрузка на команду: алертов в день, сколько аналитиков нужно
8. Жадный отбор правил на val → проверка на test
9. Итоги и что ждём от модели

In [ ]:
# ============================================================
# 0. НАСТРОЙКА
# ============================================================
import sys, gc, warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "data_loader.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import (
    COL_TARGET, COL_LAUND_TYPE, FEATURES_DIR, STRUCTURING_THRESHOLD,
    cache_path, memory_mb,
)
from src.eda_utils import save_fig
from src.rules import (
    default_rule_book, Rule, fit_rules, apply_rules, risk_score, any_rule,
    rule_metrics, score_curve, recall_by_typology, daily_alerts, workload_report,
)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
FEAT_PATH = FEATURES_DIR / "features_full.parquet"
print("Матрица признаков:", FEAT_PATH)

---
# 1. Загрузка данных

Из полной матрицы (9.5 млн строк) нам нужны только **val и test**: на train
правила не настраивают (у них нет параметров, кроме порогов, а пороги мы будем
выбирать на val). Читаем файл по частям — целиком он весит 932 МБ.

In [ ]:
# ============================================================
# 1. ЧИТАЕМ НУЖНЫЕ ПЕРИОДЫ ИЗ МАТРИЦЫ ПРИЗНАКОВ
# ============================================================
import pyarrow.parquet as pq

RULES_BOOK = default_rule_book()
# Какие колонки нужны: метка периода + всё, что используют правила + пара для графиков
NEED_COLS = ["period", COL_TARGET]
for _r in RULES_BOOK:
    NEED_COLS += [c for c in _r.needs if c not in NEED_COLS]
NEED_COLS = [c for c in dict.fromkeys(NEED_COLS)]          # убираем дубли, сохраняем порядок
print(f"Читаем {len(NEED_COLS)} колонок: {NEED_COLS}")

def load_periods(periods=("val", "test"), need_cols=NEED_COLS, verbose=True):
    '''
    Читает из parquet только нужные периоды, по частям (row group за row group).

    Зачем так: файл на 9.5 млн строк весит 932 МБ, а нам нужна треть. Читать
    весь файл ради трети — разбазаривать память, которая ещё пригодится.
    Одновременно запоминаем номера строк: по ним потом возьмём метки
    (типология отмывания в матрицу признаков намеренно не входила).
    '''
    pf = pq.ParquetFile(FEAT_PATH)
    parts, rows_idx = [], []
    offset = 0
    for gi in range(pf.metadata.num_row_groups):
        chunk = pf.read_row_group(gi, columns=need_cols).to_pandas()
        mask = chunk["period"].isin(periods).to_numpy()
        if mask.any():
            parts.append(chunk[mask])
            rows_idx.append(np.flatnonzero(mask) + offset)
        offset += len(chunk)
        del chunk
    X = pd.concat(parts, ignore_index=True)
    idx = np.concatenate(rows_idx)
    del parts, rows_idx
    gc.collect()
    if verbose:
        print(f"Прочитано {len(X):,} строк | память {memory_mb(X):,.0f} MB")
    return X, idx

X, row_idx = load_periods(("train", "val", "test"))
print(X["period"].value_counts().to_frame("строк"))

In [ ]:
# ---- Метки и типологии берём из подготовленного кэша по номерам строк ----
# ВАЖНО: это возможно только потому, что сборка признаков (Этап 2) сохраняла
# строки в исходном порядке. Сверяем метку Is_laundering из двух источников —
# если совпало построчно, значит склейка по позиции корректна.
labels = pd.read_parquet(cache_path(), columns=[COL_TARGET, COL_LAUND_TYPE, "Date"])
labels = labels.iloc[row_idx].reset_index(drop=True)

assert len(labels) == len(X), "Длины не совпали — склейка по позициям невозможна"
assert (labels[COL_TARGET].to_numpy() == X[COL_TARGET].to_numpy()).all(), "Метки не совпали построчно: порядок строк нарушен"
print("Сверка меток пройдена: порядок строк совпадает построчно")

y = X[COL_TARGET].to_numpy()
typology = labels[COL_LAUND_TYPE].to_numpy()
dates = labels["Date"].to_numpy()
period = X["period"].to_numpy()
offtrain_np = (period == "val") | (period == "test")
del labels
gc.collect()

print(f"Отмываний: {y.sum():,} | base rate {y.mean() * 100:.4f}%")
print(f"Будем считать метрики на val + test: {int(offtrain_np.sum()):,} строк")
print(f"Периоды: val {int((period == 'val').sum()):,}, test {int((period == 'test').sum()):,}")

---
# 2. Справочник правил

Каждое правило — это численное выражение конкретной типологии. Ниже весь набор
(он же лежит в `src/rules.py`, чтобы его можно было переиспользовать в проде
и в Streamlit-приложении).

In [ ]:
# ============================================================
# 2. СПРАВОЧНИК ПРАВИЛ
# ============================================================
book = pd.DataFrame([{"код": r.code, "правило": r.name, "типология": r.typology,
                      "условие": r.logic} for r in RULES_BOOK])
display(book)

In [ ]:
# ---- Настраиваем пороги на train ------------------------------------
# Пороги вида «аномально много» нельзя придумывать: на 9.5 млн транзакций
# «20 получателей» оказывается нормой. Берём перцентиль признака по train —
# ровно тот же принцип FIT/APPLY, что и у профилей клиентов на Этапе 2.
train_np = (X["period"].to_numpy() == "train")
RULES_BOOK = fit_rules(X[train_np], RULES_BOOK)
print("Подобранные пороги (по train-периоду):")
for _r in RULES_BOOK:
    if _r.threshold is not None:
        print(f"  {_r.code}: {_r.feature} >= {_r.threshold:,.3f}  ({_r.quantile:.0%} перцентиль)")

# ---- Применяем правила ко всем периодам ------------------------------
flags = apply_rules(X, RULES_BOOK)
score = risk_score(flags)
print(f"Таблица флагов: {flags.shape}")
print(f"Всего сработавших правил по всем строкам: {int(flags.to_numpy().sum()):,}")
print("\nСколько раз сработало каждое правило:")
display(flags.sum().sort_values(ascending=False).to_frame("сработало"))

# Train нам больше не нужен: пороги уже подобраны. Выбрасываем его, чтобы
# освободить память под дальнейшие расчёты (~450 МБ).
before = len(X)
X = X[offtrain_np].reset_index(drop=True)
flags = flags[offtrain_np].reset_index(drop=True)
score = risk_score(flags)
y, typology, dates, period = (y[offtrain_np], typology[offtrain_np],
                              dates[offtrain_np], period[offtrain_np])
train_np = (period == "train")
val_mask_np = (period == "val")
test_mask_np = (period == "test")
offtrain_np = np.ones(len(X), dtype=bool)
gc.collect()
print(f"\nОставили val + test: {len(X):,} строк из {before:,} | память {memory_mb(X):,.0f} MB")

---
# 3. Метрики каждого правила по отдельности

Смотрим на **recall** (сколько отмывания поймали) и **precision** (какая доля
алертов не мусор). Lift показывает, во сколько раз правило точнее, чем
«проверять все транзакции подряд»: lift 10 = в 10 раз концентрированнее.

In [ ]:
# ============================================================
# 3. МЕТРИКИ ПРАВИЛ (val + test; train не считаем — на нём подбирали пороги)
# ============================================================
offtrain_np = ~train_np
met = rule_metrics(flags[offtrain_np], y[offtrain_np], RULES_BOOK)
display(met[["правило", "название", "alerts", "alert_rate_%", "tp", "precision_%",
             "recall_%", "lift", "f1"]])

In [ ]:
# ---- Визуализация: recall против precision --------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

m = met.sort_values("recall_%", ascending=True)
axes[0].barh(m["правило"] + " " + m["название"].str.slice(0, 28), m["recall_%"], color="#C44E52")
axes[0].set_title("Recall: какую долю отмывания ловит правило")
axes[0].set_xlabel("% от всех отмываний")

m2 = met.sort_values("lift", ascending=True)
axes[1].barh(m2["правило"] + " " + m2["название"].str.slice(0, 28), m2["lift"], color="#4C72B0")
axes[1].axvline(1.0, color="grey", lw=1)
axes[1].set_title("Lift: во сколько раз точнее, чем проверять всё подряд")
axes[1].set_xlabel("lift к base rate")
save_fig("22_rule_recall_lift", fig)
plt.show()

---
# 4. Стратегия «хотя бы одно правило»

Это то, как обычно работает первая версия системы: транзакция попадает в очередь,
если сработало любое правило. Максимальный recall, но и мусора много.

In [ ]:
# ============================================================
# 4. «ХОТЯ БЫ ОДНО ПРАВИЛО»
# ============================================================
# Метрики считаем на val + test (train исключили: на нём подбирали пороги)
y_eval = y[offtrain_np]
alert_any = any_rule(flags[offtrain_np]).to_numpy()
tp_any = int((alert_any & (y_eval == 1)).sum())
prec_any = tp_any / max(alert_any.sum(), 1)
rec_any = tp_any / max((y_eval == 1).sum(), 1)
f1_any = 2 * prec_any * rec_any / max(prec_any + rec_any, 1e-12)

print("Стратегия «любое правило сработало»:")
print(f"  алертов            : {int(alert_any.sum()):,}")
print(f"  доля трафика       : {alert_any.mean() * 100:.3f}%")
print(f"  найдено отмываний  : {tp_any:,} из {int((y_eval == 1).sum()):,}")
print(f"  precision          : {prec_any * 100:.3f}%  (какая доля алертов — реальные)")
print(f"  recall             : {rec_any * 100:.1f}%   (какую долю отмывания поймали)")
print(f"  lift               : {prec_any / y_eval.mean():.2f}  (во сколько раз точнее, чем всё подряд)")
print(f"  F1                 : {f1_any:.4f}")

print("\nДля сравнения — три «глупых» стратегии:")
print(f"  проверять всё подряд : recall 100%, precision {y.mean() * 100:.3f}%, "
      f"алертов {len(y):,}")
rng = np.random.default_rng(RANDOM_STATE)
rnd = rng.random(len(y_eval)) < 0.01
print(f"  случайный 1% трафика : recall {rnd[y_eval == 1].mean() * 100:.1f}%, "
      f"precision {y_eval[rnd].mean() * 100:.3f}%")
big = (X["amount_above_threshold"].to_numpy()[offtrain_np] == 1)
print(f"  только порог $10 000 : recall {big[y_eval == 1].mean() * 100:.1f}%, "
      f"precision {y_eval[big].mean() * 100:.3f}%, алертов {int(big.sum()):,}")

---
# 5. Оценка риска: сколько правил сработало

Вместо «сработало хоть что-то» считаем, **сколько** правил сработало, и
подбираем порог `k`. Это и есть главный бизнес-компромисс:

* маленький `k` → много алертов, высокая полнота, команда тонет;
* большой `k` → алертов мало, но половина схем проходит мимо.

**Порог выбираем на val и только потом смотрим test.** Иначе мы подгоняем
решение под ответы — и в проде результат окажется хуже.

In [ ]:
# ============================================================
# 5. КРИВАЯ «ПОРОГ ПО ЧИСЛУ ПРАВИЛ»
# ============================================================
val_mask_np = (period == "val")
test_mask_np = (period == "test")
offtrain_np = (period == "val") | (period == "test")

curve_val = score_curve(y[val_mask_np], score.to_numpy()[val_mask_np])
curve_test = score_curve(y[test_mask_np], score.to_numpy()[test_mask_np])
print("VAL:"); display(curve_val[["порог_k", "alerts", "alert_rate_%", "tp",
                                  "precision_%", "recall_%", "lift", "f1"]])
print("TEST:"); display(curve_test[["порог_k", "alerts", "alert_rate_%", "tp",
                                    "precision_%", "recall_%", "lift", "f1"]])

In [ ]:
# ---- Выбираем порог на val по максимуму F1 -------------------------
best_k = int(curve_val.loc[curve_val["f1"].idxmax(), "порог_k"])
print(f"Лучший порог по F1 на val: k = {best_k}")
row = curve_test[curve_test["порог_k"] == best_k].iloc[0]
print(f"На test при этом пороге: recall {row['recall_%']:.1f}%, "
      f"precision {row['precision_%']:.3f}%, lift {row['lift']:.1f}, "
      f"алертов {int(row['alerts']):,}")

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(curve_test["порог_k"], curve_test["recall_%"], "o-", color="#C44E52", label="recall, %")
ax1.plot(curve_test["порог_k"], curve_test["precision_%"], "s-", color="#4C72B0", label="precision, %")
ax1.set_xlabel("порог k: сколько правил должно сработать")
ax1.set_ylabel("%")
ax1.axvline(best_k, color="grey", ls="--", label=f"порог с val: k = {best_k}")
ax2 = ax1.twinx()
ax2.bar(curve_test["порог_k"], curve_test["alerts"], alpha=.15, color="grey", label="алертов")
ax2.set_ylabel("алертов (правая шкала)")
ax1.set_title("Компромисс полноты и точности: кривая порога (test)")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper right")
save_fig("23_score_curve", fig)
plt.show()

---
# 6. Recall по типологиям: где наши слепые зоны

**Самая полезная таблица этапа.** Средний recall 40 % может означать
«structuring ловим на 90 %, а Cycle — на 2 %». Для риск-менеджмента это
принципиально разные новости.

In [ ]:
# ============================================================
# 6. КАКИЕ ТИПОЛОГИИ МЫ ЛОВИМ
# ============================================================
alert_best = (score.to_numpy() >= best_k)
rec_typ = recall_by_typology(alert_best, y, typology, min_support=20)
display(rec_typ)

fig, ax = plt.subplots(figsize=(11, 6))
r = rec_typ.sort_values("recall_%")
colors = ["#55A868" if v >= 50 else "#DD8452" if v >= 20 else "#C44E52" for v in r["recall_%"]]
ax.barh(r["типология"], r["recall_%"], color=colors)
ax.axvline(50, color="grey", ls="--", lw=1)
ax.set_xlabel("доль типологии, которую поймали правила, %")
ax.set_title(f"Recall по типологиям (порог k = {best_k}, val + test)")
save_fig("24_typology_recall", fig)
plt.show()

---
# 7. Сколько это стоит команде

Метрики моделей бессмысленны без операционной реальности: у команды
комплаенса ограниченное число рук. Считаем, сколько алертов в день придёт
и сколько аналитиков нужно, чтобы их разобрать.

In [ ]:
# ============================================================
# 7. НАГРУЗКА НА КОМАНДУ
# ============================================================
wl = workload_report(alert_best, y, dates, analyst_capacity_per_day=30)
print(f"Стратегия: сработало >= {best_k} правил")
for k, v in wl.items():
    print(f"  {k:32s}: {v if isinstance(v, int) else round(v, 2)}")

per_day = daily_alerts(alert_best, dates)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(per_day.index, per_day.values, lw=1, color="#4C72B0")
ax.axhline(per_day.median(), color="#C44E52", ls="--",
           label=f"медиана {per_day.median():.0f} алертов/день")
ax.axhline(30, color="green", ls=":", label="30 алертов/день = 1 аналитик")
ax.set_title("Алертов в день")
ax.tick_params(axis="x", rotation=90)
ax.legend()
save_fig("25_alerts_per_day", fig)
plt.show()

---
# 8. Жадный отбор правил: собираем минимальный набор

Правил 13, но держать их все в проде дорого: каждое требует поддержки,
документирования и объяснения регулятору. Соберём минимальный набор, который
даёт максимум recall при заданном бюджете алертов.

**Бюджет выбираем на val** — например, «не больше 0.5 % трафика», то есть
командаPhysical сможет разобрать поток.

In [ ]:
# ============================================================
# 8. ЖАДНЫЙ ОТБОР ПРАВИЛ (на val)
# ============================================================
def greedy_select(flags: pd.DataFrame, y: np.ndarray, mask: np.ndarray,
                  budget_share: float = 0.005, max_rules: int = 6) -> list[str]:
    '''
    Жадный отбор: на каждом шаге добавляем правило, которое даёт максимальный
    прирост найденных отмываний на каждый новый алерт. Останавливаемся, когда
    вышли за бюджет алертов или набрали max_rules.

    mask — какие строки используем для отбора (это val: на test не подгоняем).
    '''
    cols = list(flags.columns)
    chosen: list[str] = []
    y_m = np.asarray(y)[mask]
    n_budget = int(len(y_m) * budget_share)
    best_state = None

    for _ in range(max_rules):
        cur = (flags.loc[mask, chosen].to_numpy().any(axis=1)
               if chosen else np.zeros(mask.sum(), dtype=bool))
        n_cur, tp_cur = int(cur.sum()), int((cur & (y_m == 1)).sum())
        cands = []
        for c in cols:
            if c in chosen:
                continue
            new = cur | flags.loc[mask, c].to_numpy()
            n_new, tp_new = int(new.sum()), int((new & (y_m == 1)).sum())
            if n_new > n_budget:                       # бюджет превышен — правило не берём
                continue
            gain, cost = tp_new - tp_cur, max(n_new - n_cur, 1)
            cands.append((gain / cost, gain, c, n_new, tp_new))
        if not cands:
            break
        cands.sort(reverse=True)
        _, gain, best_col, n_new, tp_new = cands[0]
        chosen.append(best_col)
        best_state = (n_new, tp_new)
        print(f"  + {best_col}: +{gain} найденных, алертов {n_new:,} "
              f"({n_new / len(y_m) * 100:.3f}% трафика)")
    return chosen

BUDGET = 0.005
print(f"Бюджет алертов: {BUDGET * 100:.1f}% трафика val-периода\n")
chosen = greedy_select(flags, y, val_mask_np, budget_share=BUDGET, max_rules=6)
print("\nВыбранные правила:", chosen)

In [ ]:
# ---- Проверяем выбранный набор на test (это честная проверка) -------
alert_sel_val = flags.loc[val_mask_np, chosen].to_numpy().any(axis=1)
alert_sel_test = flags.loc[test_mask_np, chosen].to_numpy().any(axis=1)

def quick_report(name, alert, mask):
    yy = y[mask]
    tp = int((alert & (yy == 1)).sum())
    prec = tp / max(alert.sum(), 1)
    rec = tp / max((yy == 1).sum(), 1)
    print(f"{name:28s} | алертов {int(alert.sum()):>7,} ({alert.mean() * 100:5.3f}% трафика) "
          f"| найдено {tp:>4,} | precision {prec * 100:5.3f}% | recall {rec * 100:5.1f}% "
          f"| lift {prec / yy.mean():5.1f}")

print("Сравнение на VAL (здесь подбирали):")
quick_report("только порог $10k", (X.loc[val_mask_np, "amount_above_threshold"].to_numpy() == 1), val_mask_np)
quick_report("все правила (любое)", any_rule(flags)[val_mask_np].to_numpy(), val_mask_np)
quick_report(f"порог k >= {best_k}", (score.to_numpy() >= best_k)[val_mask_np], val_mask_np)
quick_report("отобранный набор", alert_sel_val, val_mask_np)

print("\nСравнение на TEST (здесь НЕ подбирали — честная оценка):")
quick_report("только порог $10k", (X.loc[test_mask_np, "amount_above_threshold"].to_numpy() == 1), test_mask_np)
quick_report("все правила (любое)", any_rule(flags)[test_mask_np].to_numpy(), test_mask_np)
quick_report(f"порог k >= {best_k}", (score.to_numpy() >= best_k)[test_mask_np], test_mask_np)
quick_report("отобранный набор", alert_sel_test, test_mask_np)

In [ ]:
# ---- Recall по типологиям для отобранного набора (на test) ----------
rec_sel = recall_by_typology(alert_sel_test, y[test_mask_np],
                             typology[test_mask_np], min_support=5)
display(rec_sel)

---
# 9. Итоги Этапа 3

In [ ]:
# ============================================================
# 9. ИТОГИ
# ============================================================
summary = {
    "правил_в_справочнике": len(RULES_BOOK),
    "выбрано_правил": len(chosen),
    "порог_по_числу_правил_k": best_k,
    "recall_test_%": round(float((score.to_numpy() >= best_k)[test_mask_np][y[test_mask_np] == 1].mean() * 100), 1),
    "precision_test_%": round(float(y[test_mask_np][(score.to_numpy() >= best_k)[test_mask_np]].mean() * 100), 3),
    "алертов_в_день_медиана": round(float(per_day.median()), 1),
    "проверок_на_1_случай": round(wl["проверок_на_1_реальный_случай"], 1),
    "аналитиков_нужно": round(wl["аналитиков_нужно"], 2),
    "типологий_с_recall_>50%": int((rec_sel["recall_%"] >= 50).sum()) if len(rec_sel) else 0,
    "типологий_с_recall_<10%": int((rec_sel["recall_%"] < 10).sum()) if len(rec_sel) else 0,
}
for k, v in summary.items():
    print(f"{k:28s}: {v}")

checklist = [
    "[x] Справочник правил: каждое правило = численная типология",
    "[x] Метрики каждого правила: recall / precision / lift / алерты",
    "[x] Стратегия «любое правило» и её цена (сколько мусора)",
    "[x] Оценка риска по числу сработавших правил + выбор порога на val",
    "[x] Recall по типологиям — видно, какие схемы ловим, а какие нет",
    "[x] Нагрузка на команду: алертов в день, сколько аналитиков нужно",
    "[x] Жадный отбор минимального набора правил под бюджет алертов",
    "[x] Честная проверка на test (пороги и отбор делались только на val)",
]
print("\n".join(checklist))
print("\nЭТАП 3 ГОТОВ. Дальше: Этап 4 — ML-модель должна обогнать этот baseline.")